In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
!pip3 install tiktoken
from torch.utils.data import Dataset,DataLoader

In [ ]:
class SelfAttention(nn.Module):
  def __init__(self,embed_dim,d_out,qkv_bias=False):
    super().__init__()
    self.weight_query=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.weight_key=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.weight_values=nn.Linear(embed_dim,d_out,bias=qkv_bias)
  def forward(self,x):
    query=self.weight_query(x)
    key=self.weight_key(x)
    value=self.weight_values(x)
    attn_scores=query@key.T
    attn_weights=nn.Softmax(attn_scores/(d_out**0.5),dim=-1)
    context_vec=attn_weights@value
    return context_vec

In [9]:
class CausalAttention(nn.Module):
  def __init__(self,context_length,embed_dim,d_out,qkv_bias=False):
    super().__init__()
    self.w_key=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.w_query=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.w_value=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.dropout=nn.Dropout(0.1)
    self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))
  def forward(self,x):
    b,num_tokens,embed_shape=x.shape
    keys=self.w_key(x)
    query=self.w_query(x)
    values=self.w_value(x)
    attn_scores=query@keys.transpose(1,2)
    attn_scores.masked_fill(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
    attn_weights=torch.softmax(attn_scores/(d_out**0.5),dim=-1)
    attn_weights=self.dropout(attn_weights)
    context_vec=attn_weights@values
    return context_vec

In [10]:
class MultiAttentionHeadWrapper(nn.Module):
  def __init__(self,context_length,embed_dim,d_out,num_heads,qkv_bias=False):
    super().__init__()
    self.heads=nn.ModuleList([CausalAttention(context_length,embed_dim,d_out,qkv_bias) for _ in range(num_heads)])
  def forward(self,x):
    return torch.cat([head(x) for head in self.heads],dim=-1)

In [4]:
class MultiHeadAttention(nn.Module):
  def __init__(self,context_length,embed_dim,d_out,num_heads,dropout,qkv_bias=False):
    super().__init__()
    self.d_out=d_out
    self.head_dim=d_out/num_heads
    self.num_heads=num_heads
    self.w_query=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.w_key=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.w_value=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))
  def forward(self,x):
    b,num_tokens,embed_shape=x.shape
    query=self.w_query(x)
    key=self.w_key(x)
    value=self.w_value(x)       ### (b,num_tokens,d_out)###

    query=query.view(b,num_tokens,self.num_heads,self.head_dim)
    key=key.view(b,num_tokens,self.num_heads,self.head_dim)
    value=value.view(b,num_tokens,self.num_heads,self.head_dim)       ###(b,num_tokens,num_heads,head_dim)###

    query=query.transpose(1,2)
    key=key.transpose(1,2)
    value=value.transpose(1,2)          ###(b,num_heads,num_tokens,head_dim)

    attn_scores=query@key.transpose(2,3)
    attn_scores.masked_fill(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
    attn_weights=torch.softmax((attn_scores/(self.head_dim**0.5)),dim=-1)
    attn_weights=self.dropout(attn_weights)
    context_vec=attn_weights@value                ### (b,num_heads,num_tokens,num_tokens)@(b,num_heads,num_tokens,head_dim)

    context_vec.transpose(1,2)                    ###(b,num_tokens,num_heads,head_dim)

    context_vec=context_vec.contiguous().view(b,num_tokens,self.d_out)
    return context_vec
